# PSORTb CTS Demo

End-to-end demo of the cdm_psortb CTS tool.

- **Image:** `ghcr.io/kbaseincubator/cdm_psortb:0.1.3@sha256:09389382bf00e4759bbc6223b751493b3e65c397771b0749ef2aa6cb422165b2`
- **Refdata:** none (PSORTb's models are bundled in the image)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/psortb/demo_20261106`

PSORTb predicts subcellular localization of bacterial and archaeal proteins. Input is **protein FASTA** (`.faa`). Each run needs the right organism model: `-n` (Gram-negative), `-p` (Gram-positive), or `-a` (archaea). Genomes are therefore grouped by type and each group is submitted as its own job.

The wrapper runs psort once per input file and writes `/out/<input-basename>.psortb.tsv` plus a `.psortb.log`. Since 0.1.3 it fails the job (non-zero exit) on an input with no sequences, an empty result, or a terse result whose row count differs from the input.

Re-executed 2026-11-06 with 0.1.3. The earlier run used 0.1.2 and the Gram-negative model for all four genomes, although three of them are archaea or Gram-positive.

## 1. Setup

In [1]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_psortb:0.1.3@sha256:09389382bf00e4759bbc6223b751493b3e65c397771b0749ef2aa6cb422165b2"
OUTPUT_DIR = "cts/io/jplfaria/output/psortb/demo_20261106"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': ['cts_user'], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List protein FASTA inputs and group them by organism type

The four protein `.faa` files from the checkm2 demo (`cts/io/gavin/test_output/kb_demo/...`). Two are archaea, one is Gram-positive, one is Gram-negative.

In [2]:
ORGANISM = {   # accession -> (organism, PSORTb model flag)
    "GCA_000008085.1": ("Nanoarchaeum equitans (archaeon)", "-a"),
    "GCA_000145985.1": ("Ignisphaera aggregans (archaeon)", "-a"),
    "GCA_000010565.1": ("Pelotomaculum thermopropionicum (Gram-positive)", "-p"),
    "GCA_000147015.1": ("Candidatus Zinderia insecticola (Gram-negative)", "-n"),
}
input_files = sorted(
    f"cts/{o.object_name}"
    for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True)
    if o.object_name.endswith(".faa")
)
groups, n_input = {}, {}
for f in input_files:
    acc = f.split("/")[-1][:15]
    name, flag = ORGANISM[acc]
    groups.setdefault(flag, []).append(f)
    raw = mincli.get_object(*f.split("/", 1)).read().decode("utf-8")
    n_input[f] = sum(1 for line in raw.splitlines() if line.startswith(">"))
    print(f"  {flag}  {n_input[f]:>4} proteins  {name:48s} {f.split('/')[-1]}")


  -a   583 proteins  Nanoarchaeum equitans (archaeon)                 GCA_000008085.1_ASM808v1_genomic.faa
  -p  3001 proteins  Pelotomaculum thermopropionicum (Gram-positive)  GCA_000010565.1_ASM1056v1_genomic.faa
  -a  1999 proteins  Ignisphaera aggregans (archaeon)                 GCA_000145985.1_ASM14598v1_genomic.faa
  -n   219 proteins  Candidatus Zinderia insecticola (Gram-negative)  GCA_000147015.1_ASM14701v1_genomic.faa


## 3. Submit one PSORTb job per organism type

Args sent to the wrapper:
- the organism model flag for the group (`-a`, `-p`, or `-n`)
- `-o terse`: tab-separated output (`SeqID`, `Localization`, `Score`)
- input files via `tscli.insert_files()`, after the flags

Do **not** pass `-i`, `-r`, or `-x`; the wrapper handles input and output paths. `num_containers=len(files)` runs the files of a group in parallel.

In [3]:
jobs = {}
for flag, files in groups.items():
    jobs[flag] = tscli.submit_job(
        IMAGE,
        files,
        f"{OUTPUT_DIR}/{flag.strip('-')}",
        cluster="kbase",
        declobber=True,
        output_mount_point="/out",
        args=[
            flag,                       # organism model for this group
            "-o", "terse",              # tab-separated: SeqID, Localization, Score
            tscli.insert_files(),       # input files after the flags
        ],
        num_containers=len(files),
        cpus=2,
        memory="4GB",
        runtime="PT1H",
    )
    print(f"{flag}: job {jobs[flag].id} ({len(files)} file(s))")


-a: job 8c8e988f-b019-4f4d-8b50-4067a5c662fb (2 file(s))
-p: job cdd24918-f0a2-4038-9b06-2bb91f9968dc (1 file(s))


-n: job 5191199d-d077-4827-a8d4-5e5f1aa1989d (1 file(s))


## 4. Wait for completion

In [4]:
# CTS keeps container logs only for failed jobs; the checks below validate the outputs.
for flag, j in jobs.items():
    j.wait_for_completion()
    print(f"{flag}: state={j.get_job_status()['state']}  exit codes={j.get_exit_codes().get('exit_codes')}")


-a: state=complete  exit codes=[0, 0]


-p: state=complete  exit codes=[0]


-n: state=complete  exit codes=[0]


## 5. Inspect output files

In [5]:
outs = [(flag, o) for flag, j in jobs.items() for o in j.get_job()["outputs"]]
print(f"{len(outs)} output files:")
for flag, o in outs:
    print(f"  {flag}  {o['file']}")


8 output files:
  -a  cts/io/jplfaria/output/psortb/demo_20261106/a/0/GCA_000008085.1_ASM808v1_genomic.psortb.log
  -a  cts/io/jplfaria/output/psortb/demo_20261106/a/0/GCA_000008085.1_ASM808v1_genomic.psortb.tsv
  -a  cts/io/jplfaria/output/psortb/demo_20261106/a/1/GCA_000145985.1_ASM14598v1_genomic.psortb.log
  -a  cts/io/jplfaria/output/psortb/demo_20261106/a/1/GCA_000145985.1_ASM14598v1_genomic.psortb.tsv
  -p  cts/io/jplfaria/output/psortb/demo_20261106/p/0/GCA_000010565.1_ASM1056v1_genomic.psortb.log
  -p  cts/io/jplfaria/output/psortb/demo_20261106/p/0/GCA_000010565.1_ASM1056v1_genomic.psortb.tsv
  -n  cts/io/jplfaria/output/psortb/demo_20261106/n/0/GCA_000147015.1_ASM14701v1_genomic.psortb.log
  -n  cts/io/jplfaria/output/psortb/demo_20261106/n/0/GCA_000147015.1_ASM14701v1_genomic.psortb.tsv


## 6. Read PSORTb results into a dataframe

One `.psortb.tsv` per input file, columns `SeqID`, `Localization`, `Score`, with a header row. `SeqID` is the full FASTA header line, so the ID is taken as its first word.

In [6]:
import io
import pandas as pd

frames = []
for flag, o in outs:
    if not o["file"].endswith(".psortb.tsv"):
        continue
    raw = mincli.get_object(*o["file"].split("/", 1)).read().decode("utf-8", errors="replace")
    df = pd.read_csv(io.StringIO(raw), sep="\t")
    df["protein_id"] = df["SeqID"].str.split().str[0]
    df["model"] = flag
    df["source_file"] = o["file"].split("/")[-1].replace(".psortb.tsv", "")
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total rows: {len(all_df):,}\n")
print(pd.crosstab(all_df["source_file"], all_df["Localization"]).to_string())
all_df.head(10)


Total rows: 5,802

Localization                        Cellwall  Cytoplasmic  CytoplasmicMembrane  Extracellular  Unknown
source_file                                                                                           
GCA_000008085.1_ASM808v1_genomic           1          397                   83              5       97
GCA_000010565.1_ASM1056v1_genomic         27         1782                  682             22      488
GCA_000145985.1_ASM14598v1_genomic         6         1345                  402             11      235
GCA_000147015.1_ASM14701v1_genomic         0          160                   19              1       39


,SeqID,Localization,Score,protein_id,model,source_file
0,AE017199.1_1 # 1 # 879 # -1 # ID=1_1;partial=1...,Cytoplasmic,7.50,AE017199.1_1,-a,GCA_000008085.1_ASM808v1_genomic
1,AE017199.1_2 # 883 # 2691 # 1 # ID=1_2;partial...,Cytoplasmic,9.96,AE017199.1_2,-a,GCA_000008085.1_ASM808v1_genomic
2,AE017199.1_3 # 2668 # 3189 # -1 # ID=1_3;parti...,Cytoplasmic,7.50,AE017199.1_3,-a,GCA_000008085.1_ASM808v1_genomic
3,AE017199.1_4 # 3290 # 3748 # -1 # ID=1_4;parti...,Cytoplasmic,7.50,AE017199.1_4,-a,GCA_000008085.1_ASM808v1_genomic
4,AE017199.1_5 # 3745 # 4665 # -1 # ID=1_5;parti...,Unknown,2.50,AE017199.1_5,-a,GCA_000008085.1_ASM808v1_genomic
5,AE017199.1_6 # 4680 # 5090 # 1 # ID=1_6;partia...,Cytoplasmic,9.96,AE017199.1_6,-a,GCA_000008085.1_ASM808v1_genomic
6,AE017199.1_7 # 5028 # 5561 # -1 # ID=1_7;parti...,CytoplasmicMembrane,9.99,AE017199.1_7,-a,GCA_000008085.1_ASM808v1_genomic
7,AE017199.1_8 # 5537 # 5707 # -1 # ID=1_8;parti...,Cytoplasmic,7.50,AE017199.1_8,-a,GCA_000008085.1_ASM808v1_genomic
8,AE017199.1_9 # 5759 # 7000 # 1 # ID=1_9;partia...,Cytoplasmic,7.50,AE017199.1_9,-a,GCA_000008085.1_ASM808v1_genomic
9,AE017199.1_10 # 7105 # 7401 # -1 # ID=1_10;par...,Cytoplasmic,7.50,AE017199.1_10,-a,GCA_000008085.1_ASM808v1_genomic


## 7. End-to-end check

Every input file must have a result with exactly one row per input protein.

In [7]:
rows = all_df.groupby("source_file").size()
expected = {f.split("/")[-1].rsplit(".", 1)[0]: n for f, n in n_input.items()}
checks = {
    "all jobs complete, all exit 0": all(j.get_job_status()["state"] == "complete"
                                         and set(j.get_exit_codes().get("exit_codes", [])) == {0} for j in jobs.values()),
    "one result per input file": set(rows.index) == set(expected),
    "rows == input proteins, per file": all(rows.get(k) == v for k, v in expected.items()),
    "every protein ID appears once per file": not all_df.duplicated(["source_file", "protein_id"]).any(),
    "real localizations present": all_df["Localization"].notna().all() if len(all_df) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off, inspect outputs above.")


  [x] all jobs complete, all exit 0
  [x] one result per input file
  [x] rows == input proteins, per file
  [x] every protein ID appears once per file
  [x] real localizations present

All green.
